In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
constraints = """torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
transformers>=4.48,<=4.57
"""
with open("/content/constraints.txt", "w") as f:
    f.write(constraints)

In [ ]:
!wget -qO- https://astral.sh/uv/install.sh | sh

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!uv pip install --system torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

!uv pip install --system \
    sympy numpy transformers vllm tqdm \
    antlr4-python3-runtime==4.11.1 accelerate \
    -c /content/constraints.txt

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/usr/local/bin/uv
uv 0.11.8 (x86_64-unknown-linux-gnu)


In [ ]:
# If vllm import fails the first time after install, restart the session and re-run all cells. Same for any persistent setup errors.
import torch
import vllm
print("Post-restart checks:")
print("  torch:", torch.__version__)
print("  CUDA available:", torch.cuda.is_available())
print("  Device:", torch.cuda.get_device_name(0))
print("  vllm:", vllm.__version__)
print("  GPU memory free:", round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), "GB")

Post-restart checks:
  torch: 2.5.1+cu121
  CUDA available: False


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
from pathlib import Path
import sys
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/cse151b")
GIVEN_DATA_DIR = PROJECT_DIR / "given_data"
OUTPUT_DIR = PROJECT_DIR  # final outputs land here on Drive

# Local hot-path directory (Colab disk — fast, reliable, but wiped on runtime death)
LOCAL_DIR = Path("/content/local_results")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Inputs
# DATA_PATH = str(GIVEN_DATA_DIR / "public.jsonl")
DATA_PATH = str(GIVEN_DATA_DIR / "private.jsonl")

# Hot-path outputs — written to local disk during generation
RESPONSES_PATH = LOCAL_DIR / "responses.jsonl"
LOG_PATH = LOCAL_DIR / "responses.log"

# Drive backup of responses (snapshot target during run, restore source after restart)
RESPONSES_BACKUP = OUTPUT_DIR / "responses.jsonl"

# Final outputs — written to Drive (only one write each, at the end of their step)
SCORED_PATH = OUTPUT_DIR / "scored_results.jsonl"
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

# Make given_data importable
sys.path.insert(0, str(GIVEN_DATA_DIR))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert OUTPUT_DIR.exists(), f"Drive not mounted? {OUTPUT_DIR} missing"

# If a Drive backup exists from a previous session but no local copy, restore it
if RESPONSES_BACKUP.exists() and not RESPONSES_PATH.exists():
    shutil.copy2(RESPONSES_BACKUP, RESPONSES_PATH)
    print(f"Restored {RESPONSES_PATH.stat().st_size} bytes from Drive backup")

In [ ]:
import csv
import json
import re
import time
from collections import Counter
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
MAX_TOKENS = 16384

# Set GPU_ID earlier if training off of colab and have multiple GPUs. Need to declare before torch is imported.
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

In [ ]:
with open(DATA_PATH) as f:
    data = [json.loads(line) for line in f]

In [ ]:
n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

In [ ]:
# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "If the problem has multiple sub-answers, place them inside a single \\boxed{} "
    "separated by commas, e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices, then select the single best option. "
    "Please reason step by step, and put only the letter of your chosen option "
    "within \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [ ]:
# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",                   # drop bnb on A100 — BF16 is faster AND better quality
    trust_remote_code=True,
    max_model_len=16384,
    gpu_memory_utilization=0.92,
    max_num_batched_tokens=24576,       # match max_model_len; 32768 was overprovisioned
    max_num_seqs=64,                   # 256 is fine if you have headroom, 128 is safer. Tests set OOM on 128.
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
    disable_log_stats=True,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,                   # tame the long-tail thinking traces
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed = 5,
    stop=None,
)

print("Model loaded.")

In [ ]:
import utils
from utils import last_boxed_only_string, remove_boxed
from judger import Judger

In [ ]:
def extract_letter(text: str) -> str:
    # Strip thinking trace — only look at content after </think>
    think_end = text.rfind("</think>")
    search_text = text[think_end + len("</think>"):] if think_end >= 0 else text

    # First try: pull the last \boxed{...} content using utils' brace-aware parser
    boxed = last_boxed_only_string(search_text)
    if boxed is not None:
        inner = remove_boxed(boxed)
        if inner:
            m = re.search(r"[A-Za-z]", inner)
            if m:
                return m.group(0).upper()
    # Fallback: last standalone capital letter in the post-think response
    matches = re.findall(r"\b([A-Z])\b", search_text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

In [ ]:
judger = Judger(strict_extract=False)
print("Scoring helpers ready.")

In [ ]:
#
#
# NEW EXPERIMENT TESTING BEGINS HERE
#
# Besides changing model params
#
#

In [ ]:
# Use a small held-out subset for fast iteration
EXPERIMENT_DIR = Path("/content/drive/MyDrive/cse151b/experiments")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
NUM_SAMPLES = 20

# Pick first NUM_SAMPLES items for fast experiments — adjust to taste
EVAL_SUBSET = data[:NUM_SAMPLES]
print(f"Experiment subset: {len(EVAL_SUBSET)} items")

def extract_freeform_answer(text):
    """Pull the last boxed answer out of a response (post-think)."""
    think_end = text.rfind("</think>")
    search = text[think_end + len("</think>"):] if think_end >= 0 else text
    boxed = last_boxed_only_string(search)
    if boxed:
        return remove_boxed(boxed) or ""
    return ""

def evaluate(items, responses_per_item):
    """Score a list of (item, response) pairs. responses_per_item can be a single response or a list."""
    n_correct = 0
    details = []
    for item, resp in zip(items, responses_per_item):
        if isinstance(resp, list):
            # Self-consistency: vote across responses
            answers = [extract_freeform_answer(r) if not item.get("options")
                       else extract_letter(r) for r in resp]
            # Majority vote, ignoring empty answers
            non_empty = [a for a in answers if a]
            if not non_empty:
                pred, correct = "", False
            else:
                pred = Counter(non_empty).most_common(1)[0][0]
                if item.get("options"):
                    correct = pred == str(item["answer"]).strip().upper()
                else:
                    gold = item["answer"]
                    gold_list = gold if isinstance(gold, list) else [gold]
                    try:
                        correct = judger.auto_judge(pred=f"\\boxed{{{pred}}}", gold=gold_list,
                                                    options=[[]] * len(gold_list))
                    except Exception:
                        correct = False
            details.append({"id": item.get("id"), "pred": pred, "correct": correct, "n_samples": len(resp)})
        else:
            # Single response
            if item.get("options"):
                correct = score_mcq(resp, str(item["answer"]))
            else:
                gold = item["answer"]
                gold_list = gold if isinstance(gold, list) else [gold]
                try:
                    correct = judger.auto_judge(pred=resp, gold=gold_list,
                                                options=[[]] * len(gold_list))
                except Exception:
                    correct = False
            details.append({"id": item.get("id"), "correct": correct})
        n_correct += int(correct)
    return n_correct / len(items), details

def make_prompt_with_system(item, system, user_prefix=""):
    """Build a prompt with custom system + optional user prefix."""
    _, user = build_prompt(item["question"], item.get("options"))
    full_user = user_prefix + user if user_prefix else user
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": full_user}],
        tokenize=False, add_generation_prompt=True,
    )

In [ ]:
# Baseline: current production settings on the subset
# USE IF TESTING PARAMS
# sampling_base = SamplingParams(max_tokens=12288, temperature=0.6, top_p=0.95, top_k=20)

prompts = [make_prompt_with_system(item, build_prompt(item["question"], item.get("options"))[0])
           for item in EVAL_SUBSET]
outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=True)
baseline_responses = [out.outputs[0].text.strip() for out in outputs]

acc_base, _ = evaluate(EVAL_SUBSET, baseline_responses)
print(f"Baseline on {len(EVAL_SUBSET)} items: {acc_base*100:.1f}%")

In [ ]:
N_SAMPLES = 4

sampling_sc = SamplingParams(
    n=N_SAMPLES,                # vLLM samples N times in one call
    max_tokens=MAX_TOKENS,
    temperature=0.7,            # bumped from 0.6 for more diversity
    top_p=0.95,
    top_k=20,
)

prompts = []
for item in EVAL_SUBSET:
    sys_p, _ = build_prompt(item["question"], item.get("options"))
    prompts.append(make_prompt_with_system(item, sys_p))

print(f"Generating {N_SAMPLES} samples × {len(prompts)} prompts...")
outputs = llm.generate(prompts, sampling_params=sampling_sc, use_tqdm=True)

# Each output has N completions in .outputs
responses_grouped = [[o.text.strip() for o in out.outputs] for out in outputs]

acc_sc, details_sc = evaluate(EVAL_SUBSET, responses_grouped)
print(f"\nSelf-consistency (n={N_SAMPLES}): {acc_sc*100:.1f}%")

# Save for analysis
with open(EXPERIMENT_DIR / f"self_consistency_n{N_SAMPLES}.jsonl", "w") as f:
    for d in details_sc:
        f.write(json.dumps(d) + "\n")

In [ ]:
# Pick 2 worked examples — ideally items you've seen the model handle correctly
# These are placeholders; replace with real items + correct solutions from your data
FEWSHOT_FREEFORM = """Example 1:
Question: What is the sum of the first 5 positive integers?
Solution: We compute 1 + 2 + 3 + 4 + 5 = 15. The answer is \\boxed{15}.

Example 2:
Question: If x + 3 = 10, what is 2x?
Solution: From x + 3 = 10, we get x = 7. So 2x = 14. The answer is \\boxed{14}.

Now solve this problem:
"""

FEWSHOT_MCQ = """Example 1:
Question: What is 2 + 2?

Options:
A. 3
B. 4
C. 5
D. 6

Answer: \\boxed{B}

Now answer this question:
"""

prompts = []
for item in EVAL_SUBSET:
    sys_p, _ = build_prompt(item["question"], item.get("options"))
    prefix = FEWSHOT_MCQ if item.get("options") else FEWSHOT_FREEFORM
    prompts.append(make_prompt_with_system(item, sys_p, user_prefix=prefix))

sampling_fs = SamplingParams(
    max_tokens=12288, temperature=0.6, top_p=0.95, top_k=20,
)
outputs = llm.generate(prompts, sampling_params=sampling_fs, use_tqdm=True)
responses = [out.outputs[0].text.strip() for out in outputs]

acc_fs, details_fs = evaluate(EVAL_SUBSET, responses)
print(f"\nFew-shot: {acc_fs*100:.1f}%")

with open(EXPERIMENT_DIR / "fewshot.jsonl", "w") as f:
    for d in details_fs:
        f.write(json.dumps(d) + "\n")

In [ ]:
REFLECT_SYSTEM = (
    "You are an expert mathematician verifying a previous solution. "
    "Read the problem and the candidate solution. If the solution is correct, restate the answer in \\boxed{}. "
    "If you find an error, solve it correctly and put your final answer in \\boxed{}."
)

# Step 1: get initial answers
sampling_init = SamplingParams(max_tokens=12288, temperature=0.6, top_p=0.95, top_k=20)
init_prompts = [make_prompt_with_system(item, build_prompt(item["question"], item.get("options"))[0])
                for item in EVAL_SUBSET]
init_outputs = llm.generate(init_prompts, sampling_params=sampling_init, use_tqdm=True)
init_responses = [out.outputs[0].text.strip() for out in init_outputs]

# Step 2: ask for verification
verify_prompts = []
for item, init_resp in zip(EVAL_SUBSET, init_responses):
    _, original_user = build_prompt(item["question"], item.get("options"))
    verify_user = (f"Problem:\n{original_user}\n\n"
                   f"Candidate solution:\n{init_resp[-2000:]}\n\n"  # last 2k chars to avoid context blow-up
                   f"Verify the answer. If wrong, solve correctly. Final answer in \\boxed{{}}.")
    verify_prompts.append(tokenizer.apply_chat_template(
        [{"role": "system", "content": REFLECT_SYSTEM},
         {"role": "user", "content": verify_user}],
        tokenize=False, add_generation_prompt=True,
    ))

verify_outputs = llm.generate(verify_prompts, sampling_params=sampling_init, use_tqdm=True)
verify_responses = [out.outputs[0].text.strip() for out in verify_outputs]

acc_reflect, details_reflect = evaluate(EVAL_SUBSET, verify_responses)
print(f"\nReflection: {acc_reflect*100:.1f}%")

# Compare against baseline (no reflection) on same items
acc_base, _ = evaluate(EVAL_SUBSET, init_responses)
print(f"  Baseline on same items: {acc_base*100:.1f}%")
print(f"  Delta: {(acc_reflect - acc_base)*100:+.1f}%")

with open(EXPERIMENT_DIR / "reflection.jsonl", "w") as f:
    for d in details_reflect:
        f.write(json.dumps(d) + "\n")

In [ ]:
CoT_SYSTEM = (
    "You are an expert mathematician. Think step by step, showing all reasoning. "
    "After each major step, briefly verify it before continuing. "
    "Put your final answer in \\boxed{}."
)

prompts = []
for item in EVAL_SUBSET:
    sys_p_to_use = CoT_SYSTEM
    if item.get("options"):
        # For MCQ, append the MCQ instruction
        sys_p_to_use = CoT_SYSTEM + " For multiple choice, output only the letter inside \\boxed{}."
    prompts.append(make_prompt_with_system(item, sys_p_to_use))

outputs = llm.generate(prompts, sampling_params=sampling_init, use_tqdm=True)
responses = [out.outputs[0].text.strip() for out in outputs]

acc_cot, _ = evaluate(EVAL_SUBSET, responses)
print(f"\nCoT prompt variant: {acc_cot*100:.1f}%")

In [ ]:
print("=" * 50)
print("EXPERIMENT SUMMARY")
print("=" * 50)

# Run baseline if you don't have it
print(f"  Baseline (single, T=0.6):   {acc_base*100:5.1f}%")
print(f"  Self-consistency (n=4):     {acc_sc*100:5.1f}%")
print(f"  Few-shot:                   {acc_fs*100:5.1f}%")
print(f"  Reflection:                 {acc_reflect*100:5.1f}%")
print(f"  CoT prompt variant:         {acc_cot*100:5.1f}%")